In [10]:
import json
import os
from collections.abc import Callable
from typing import Any

from dotenv import load_dotenv
from google.genai import Client

load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
google_client = Client(api_key=GOOGLE_API_KEY)
MODEL = "gemini-3.5-flash-lite"

### Task 1: Core Agent Loop

In [11]:
def get_stock_price(ticker: str) -> float:
    prices = {
        "AAPL": 235.50,
        "GOOGL": 188.00,
        "MSFT": 430.00,
        "AMZN": 195.00,
    }
    symbol = ticker.upper()
    if symbol not in prices:
        raise ValueError(f"Unknown stock ticker: {ticker}")
    return prices[symbol]


def convert_currency(amount: float, from_currency: str, to_currency: str) -> float:
    rates_to_usd = {
        "USD": 1.0,
        "EUR": 1.08,
        "GBP": 1.30,
        "INR": 0.012,
    }
    src = from_currency.upper()
    dst = to_currency.upper()
    if src not in rates_to_usd or dst not in rates_to_usd:
        raise ValueError(f"Unsupported currency: {src} or {dst}")
    return round((amount * rates_to_usd[src]) / rates_to_usd[dst], 2)


TOOL_DEFINITIONS = [
    {
        "type": "function",
        "name": "get_stock_price",
        "description": "Retrieve current stock price for a given ticker symbol in USD.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {"type": "string", "description": "Stock ticker symbol, e.g. AAPL"},
            },
            "required": ["ticker"],
        },
    },
    {
        "type": "function",
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number", "description": "Amount to convert"},
                "from_currency": {"type": "string", "description": "Source currency, e.g. USD"},
                "to_currency": {"type": "string", "description": "Target currency, e.g. EUR"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
]

TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "get_stock_price": get_stock_price,
    "convert_currency": convert_currency,
}

In [12]:
async def run_agent(
    goal: str,
    client: Any,
    model: str,
    max_steps: int = 5,
) -> str:
    """Core agent loop: observe -> LLM -> execute tools -> append results."""
    # 1. Observe: Initialize agent loop with the user goal
    interaction = client.interactions.create(
        model=model,
        input=goal,
        tools=TOOL_DEFINITIONS,
    )

    for _ in range(max_steps):
        # 2. Extract tool calls requested by the LLM
        tool_calls = [step for step in (interaction.steps or []) if step.type == "function_call"]

        # Final-answer exit condition: no tool calls means the model produced final answer
        if not tool_calls:
            return interaction.output_text or ""

        # 3. Execute tools
        tool_results = []
        for call in tool_calls:
            result = TOOL_REGISTRY[call.name](**(call.arguments or {}))
            tool_results.append(
                {
                    "type": "function_result",
                    "name": call.name,
                    "call_id": call.id,
                    "result": [{"type": "text", "text": json.dumps({"result": result})}],
                }
            )

        # 4. Append results & observe next LLM output
        interaction = client.interactions.create(
            model=model,
            previous_interaction_id=interaction.id,
            input=tool_results,
            tools=TOOL_DEFINITIONS,
        )

    raise RuntimeError("Agent exceeded max steps without reaching a final answer.")

In [13]:
result = await run_agent(
    goal="Get the stock price of AAPL. and tell me how much in indian rupees",
    client=google_client,
    model=MODEL,
)

print("\nFINAL ANSWER:")
print(result)


FINAL ANSWER:
The current stock price of Apple Inc. (AAPL) is **$235.50 USD**. 

Converted to Indian Rupees (INR), that is approximately **₹19,625.00 INR** (based on current exchange rates).
